# 📚 AI Literature Review Agent
### Powered by LLaMA 3.1 8B (Groq) | Built by Aigenthix

---
**A world-class AI agent acting as Research Scientist, Academic Reviewer, and Literature Review Expert.**

Steps in this notebook:
1. Install all dependencies
2. Configure Groq API with LLaMA 3.1
3. Build the 6-step AI agent pipeline
4. Launch professional Gradio UI


## 🔧 Step 1: Install Dependencies


In [ ]:
# Step 1: Install all required packages
!pip install -q groq gradio PyPDF2 langchain langchain-groq tiktoken
print('✅ Installation complete')

## 🔑 Step 2: Configure Groq API Key


In [ ]:
import os

# Option A: Colab Secrets (recommended)
# Go to 🔑 Secrets in left sidebar → Add GROQ_API_KEY
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY
    print('✅ Groq API Key loaded from Colab Secrets')
except Exception:
    # Option B: Paste directly
    GROQ_API_KEY = 'YOUR_GROQ_API_KEY_HERE'  # Replace with your key
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY
    print('⚠️ Using manually entered API key — use Colab Secrets for security')


## 🧠 Step 3: Import Libraries


In [ ]:
import os, re, json, time, io
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass

import groq
import gradio as gr
import PyPDF2

print('✅ All libraries imported')
print(f'   Groq: {groq.__version__}')
print(f'   Gradio: {gr.__version__}')


## ⚙️ Step 4: Groq LLM Client


In [ ]:
client = groq.Groq(api_key=os.environ['GROQ_API_KEY'])
MODEL = 'llama-3.1-8b-instant'

def call_groq(system_prompt: str, user_prompt: str,
              max_tokens: int = 4096, temperature: float = 0.3) -> str:
    """Call Groq LLaMA 3.1 8B with retry logic."""
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_prompt}
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                return f'[ERROR] Failed after 3 attempts: {str(e)}'

# Test
test = call_groq('You are a helpful assistant.', 'Reply: LLaMA connected!')
print(f'✅ Groq verified: {test[:60]}')


## 📄 Step 5: Document Parsing Engine


In [ ]:
@dataclass
class ParsedDocument:
    filename: str
    content: str
    char_count: int
    truncated: bool = False

def extract_text_from_pdf(file_path: str) -> str:
    try:
        with open(file_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            return '\n'.join(
                p.extract_text() or '' for p in reader.pages
            ).strip()
    except Exception as e:
        return f'[PDF_ERROR: {e}]'

def extract_text_from_txt(file_path: str) -> str:
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            with open(file_path, 'r', encoding=enc) as f:
                return f.read().strip()
        except:
            continue
    return '[TEXT_ERROR: Could not decode file]'

def parse_document(file_path: str, max_chars: int = 8000) -> ParsedDocument:
    filename = os.path.basename(file_path)
    ext = filename.lower().split('.')[-1]
    content = extract_text_from_pdf(file_path) if ext == 'pdf' else extract_text_from_txt(file_path)
    truncated = len(content) > max_chars
    return ParsedDocument(
        filename=filename,
        content=content[:max_chars] if truncated else content,
        char_count=min(len(content), max_chars),
        truncated=truncated
    )

print('✅ Document parser ready')


## 🤖 Step 6: AI Agent — 6-Step Pipeline


In [ ]:
SYSTEM_PROMPT = """You are a world-class Research Scientist, Academic Reviewer, and AI Literature Review Expert.
Your analyses are objective, evidence-based, structured, and academically rigorous.
Do NOT hallucinate missing details. If information is unavailable, state 'Not specified'.
Always cite specific insights from the provided text. Be concise but meaningful."""


# ── STEP 1: Understand Research Topic ──────────────────────────
def step1_understand_topic(topic: str) -> str:
    prompt = f"""Analyze this research topic deeply:

RESEARCH TOPIC: {topic}

Return in this exact format:
CORE PROBLEM: [1-2 sentences]
KEY OBJECTIVES: [3-5 bullet points]
KEYWORDS: [8-12 comma-separated keywords]
SCOPE: [1-2 sentences describing analysis scope]
RELEVANCE CRITERIA: [what makes a paper relevant to this topic]"""
    return call_groq(SYSTEM_PROMPT, prompt)


# ── STEP 2: Analyze Individual Paper ───────────────────────────
def step2_analyze_paper(content: str, filename: str, topic: str) -> str:
    prompt = f"""Analyze this research paper for a literature review on: \"{topic}\"

PAPER: {filename}
CONTENT:
---
{content}
---

Extract (use 'Not specified' if genuinely absent):
TITLE: [paper title]
AUTHORS: [author names]
YEAR: [publication year]
KEY CONTRIBUTION: [2-3 sentences on main contribution]
METHODOLOGY: [specific methods, algorithms, frameworks]
DATASET: [datasets used]
RESULTS: [key quantitative or qualitative findings]
STRENGTHS: [2-3 genuine strengths]
LIMITATIONS: [2-3 real limitations]
RELEVANCE SCORE: [0-100 with 1-sentence justification]"""
    return call_groq(SYSTEM_PROMPT, prompt, max_tokens=2048)


# ── STEP 3: Synthesize Papers ───────────────────────────────────
def step3_synthesize(analyses: List[str], topic: str) -> str:
    combined = '\n\n---NEXT PAPER---\n\n'.join(analyses)
    prompt = f"""You analyzed {len(analyses)} papers on: \"{topic}\"

ANALYSES:
{combined}

Synthesize across all papers:
COMMON THEMES: [3-5 recurring themes]
TRENDS & PATTERNS: [emerging trends]
METHODOLOGICAL DIFFERENCES: [how approaches differ]
CONSENSUS AREAS: [what papers agree on]
CONTRADICTIONS: [conflicting findings]
COLLECTIVE STRENGTHS: [shared strengths]
COLLECTIVE WEAKNESSES: [shared limitations]"""
    return call_groq(SYSTEM_PROMPT, prompt, max_tokens=2048)


# ── STEP 4: Identify Research Gaps ─────────────────────────────
def step4_identify_gaps(synthesis: str, topic: str) -> str:
    prompt = f"""Based on the synthesis of research on \"{topic}\":

{synthesis}

Identify:
RESEARCH GAPS: [5-7 specific, actionable gaps not addressed]
UNDEREXPLORED AREAS: [2-3 domains needing investigation]
METHODOLOGICAL GAPS: [missing methods or evaluation frameworks]
FUTURE DIRECTIONS: [5-7 concrete research opportunities]
OPEN QUESTIONS: [3-5 unresolved questions in the field]"""
    return call_groq(SYSTEM_PROMPT, prompt, max_tokens=2048)


# ── STEP 5: Rank Papers ─────────────────────────────────────────
def step5_rank_papers(analyses: List[str], filenames: List[str]) -> str:
    names = '\n'.join(f'{i+1}. {f}' for i, f in enumerate(filenames))
    combined = '\n\n'.join(analyses)
    prompt = f"""Rank these {len(filenames)} papers by relevance and impact:

PAPERS:
{names}

ANALYSES:
{combined}

Return:
RANK 1: [filename] — Score: X/100 — Reason: [1 sentence]
RANK 2: [filename] — Score: X/100 — Reason: [1 sentence]
(continue for all papers)

TOP INSIGHT: [The single most important finding across all papers]"""
    return call_groq(SYSTEM_PROMPT, prompt, max_tokens=1024)


# ── STEP 6: Generate Final Literature Review ────────────────────
def step6_generate_review(topic: str, topic_analysis: str,
                          synthesis: str, gaps: str, rankings: str) -> str:
    prompt = f"""Generate a complete, publication-ready academic literature review.

TOPIC: {topic}
TOPIC ANALYSIS: {topic_analysis}
SYNTHESIS: {synthesis}
GAPS & FUTURE WORK: {gaps}
PAPER RANKINGS: {rankings}

Write a structured academic literature review:

1. INTRODUCTION
[2-3 paragraphs: introduce topic, importance, scope]

2. THEORETICAL BACKGROUND
[2 paragraphs: foundational concepts and frameworks]

3. REVIEW OF EXISTING LITERATURE
[3-4 paragraphs: synthesize findings, compare approaches]

4. CRITICAL ANALYSIS
[2 paragraphs: evaluate methodologies, highlight contradictions]

5. RESEARCH GAPS AND LIMITATIONS
[2 paragraphs: what is missing, what future research must address]

6. FUTURE RESEARCH DIRECTIONS
[1-2 paragraphs: concrete recommendations]

7. CONCLUSION
[1 paragraph: summarize key insights]

Use formal academic language. Ensure logical flow between sections."""
    return call_groq(SYSTEM_PROMPT, prompt, max_tokens=4096, temperature=0.4)


print('✅ All 6 AI agent steps defined')


## 📊 Step 7: Comparative Table Generator


In [ ]:
def generate_comparative_table(analyses: List[str], filenames: List[str], topic: str) -> list:
    combined = '\n\n---\n\n'.join(
        f'File: {filenames[i]}\n{a}' for i, a in enumerate(analyses)
    )
    prompt = f"""Extract comparative data from these paper analyses on \"{topic}\".

{combined}

Return ONLY a JSON array, no markdown, no explanation:
[
  {{
    \"paper\": \"short title or filename\",
    \"methodology\": \"brief method\",
    \"key_contribution\": \"one sentence\",
    \"strength\": \"main strength\",
    \"limitation\": \"main limitation\",
    \"relevance\": \"0-100\"
  }}
]"""
    raw = call_groq(SYSTEM_PROMPT, prompt, max_tokens=2048, temperature=0.1)
    try:
        clean = re.sub(r'```json|```', '', raw).strip()
        return json.loads(clean)
    except:
        return []


def format_table_html(table_data: list) -> str:
    if not table_data:
        return "<p style='color:#888;font-family:DM Sans,sans-serif;'>Table could not be generated — API may need more tokens.</p>"

    headers = ['Paper', 'Methodology', 'Key Contribution', 'Strength', 'Limitation', 'Relevance']
    keys    = ['paper', 'methodology', 'key_contribution', 'strength', 'limitation', 'relevance']

    rows = ''
    for i, row in enumerate(table_data):
        try:
            relevance = int(str(row.get('relevance', 0)).replace('%','').strip())
        except:
            relevance = 0
        score_color = '#16a34a' if relevance >= 80 else '#d97706' if relevance >= 60 else '#dc2626'
        bg = '#f8f6f1' if i % 2 == 0 else '#ffffff'
        cells = ''
        for k in keys:
            val = str(row.get(k, '—'))
            if k == 'relevance':
                cells += f"<td style='padding:10px 14px;border-bottom:1px solid #e5e7eb;'><span style='font-weight:700;color:{score_color};font-size:14px;'>{val}%</span></td>"
            elif k == 'paper':
                cells += f"<td style='padding:10px 14px;border-bottom:1px solid #e5e7eb;font-weight:600;color:#1e3a5f;font-size:13px;'>{val}</td>"
            else:
                cells += f"<td style='padding:10px 14px;border-bottom:1px solid #e5e7eb;font-size:12px;color:#374151;'>{val}</td>"
        rows += f"<tr style='background:{bg};'>{cells}</tr>"

    header_html = ''.join(
        f"<th style='padding:12px 14px;text-align:left;background:#1e3a5f;color:white;font-size:12px;font-weight:600;letter-spacing:0.5px;border-right:1px solid #2d5a8f;'>{h}</th>"
        for h in headers
    )
    return f"""
    <div style='overflow-x:auto;border-radius:10px;box-shadow:0 2px 16px rgba(30,58,95,0.12);margin:8px 0;'>
      <table style='width:100%;border-collapse:collapse;font-family:DM Sans,sans-serif;'>
        <thead><tr>{header_html}</tr></thead>
        <tbody>{rows}</tbody>
      </table>
    </div>"""


print('✅ Comparative table generator ready')


## 🔄 Step 8: Main Orchestrator


In [ ]:
def run_literature_review_agent(
    research_topic: str,
    uploaded_files,
    additional_text: str = '',
    progress=gr.Progress()
) -> Tuple[str, str, str, str]:
    """Run all 6 agent steps. Returns (summary_md, papers_md, review_md, table_html)."""

    if not research_topic.strip():
        err = '⚠️ Please enter a research topic first.'
        return err, err, err, '<p>' + err + '</p>'

    # Parse documents
    progress(0.05, desc='📂 Parsing uploaded documents...')
    docs: List[ParsedDocument] = []

    if uploaded_files:
        for f in uploaded_files:
            path = f.name if hasattr(f, 'name') else str(f)
            docs.append(parse_document(path))

    if additional_text.strip():
        docs.append(ParsedDocument(
            filename='manual_input.txt',
            content=additional_text.strip()[:8000],
            char_count=len(additional_text.strip())
        ))

    if not docs:
        msg = '⚠️ Please upload at least one paper or paste paper content.'
        return msg, msg, msg, '<p>' + msg + '</p>'

    # STEP 1
    progress(0.10, desc='🧠 Step 1: Analyzing research topic...')
    topic_analysis = step1_understand_topic(research_topic)

    # STEP 2
    all_analyses, paper_parts = [], []
    n = len(docs)
    for i, doc in enumerate(docs):
        progress(0.15 + i/n*0.30, desc=f'📄 Step 2: Analyzing paper {i+1}/{n}: {doc.filename}')
        analysis = step2_analyze_paper(doc.content, doc.filename, research_topic)
        all_analyses.append(analysis)
        truncated_note = ' *(truncated — first 8000 chars used)*' if doc.truncated else ''
        paper_parts.append(f'### 📄 Paper {i+1}: `{doc.filename}`{truncated_note}\n\n{analysis}\n\n---\n')

    papers_md = '\n'.join(paper_parts)

    # STEP 3
    progress(0.50, desc='🔗 Step 3: Synthesizing papers...')
    synthesis = step3_synthesize(all_analyses, research_topic)

    # STEP 4
    progress(0.65, desc='🔍 Step 4: Identifying research gaps...')
    gaps = step4_identify_gaps(synthesis, research_topic)

    # STEP 5
    progress(0.75, desc='🏆 Step 5: Ranking papers...')
    filenames = [d.filename for d in docs]
    rankings = step5_rank_papers(all_analyses, filenames)

    # STEP 6
    progress(0.82, desc='✍️ Step 6: Writing final literature review...')
    final_review = step6_generate_review(research_topic, topic_analysis, synthesis, gaps, rankings)

    # TABLE
    progress(0.93, desc='📊 Generating comparative table...')
    table_data = generate_comparative_table(all_analyses, filenames, research_topic)
    table_html = format_table_html(table_data)

    progress(1.0, desc='✅ Literature review complete!')

    # Build summary markdown
    summary_md = f"""## 🧠 Literature Review Summary

### 🎯 Topic Analysis
{topic_analysis}

---

### 🔗 Cross-Paper Synthesis
{synthesis}

---

### 🔍 Research Gaps & Future Directions
{gaps}

---

### 🏆 Paper Rankings
{rankings}
"""
    return summary_md, papers_md, final_review, table_html


print('✅ Main orchestrator ready')


## 🎨 Step 9: UI Assets (CSS & HTML)


In [ ]:
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700&family=Source+Serif+4:opsz,wght@8..60,300;8..60,400;8..60,600&family=DM+Sans:wght@300;400;500;600&display=swap');

.marquee-wrapper {
    background: linear-gradient(135deg,#1e3a5f 0%,#2d5a8f 50%,#1a4a7a 100%);
    overflow: hidden; border-bottom: 3px solid #c8a96e;
    height: 44px; display: flex; align-items: center;
}
.marquee-content {
    display: inline-flex; animation: marquee 30s linear infinite;
    white-space: nowrap; align-items: center;
}
.marquee-content span {
    font-family: 'DM Sans', sans-serif; font-size: 12px; font-weight: 500;
    letter-spacing: 2.5px; text-transform: uppercase; color: #e8d5a3; padding: 0 60px;
}
.mq-dot { color: #c8a96e; }
@keyframes marquee { 0%{transform:translateX(0)} 100%{transform:translateX(-50%)} }

.gradio-container {
    background: #f8f6f1 !important;
    font-family: 'DM Sans', sans-serif !important;
    max-width: 1400px !important; margin: 0 auto !important;
}
.hero-header {
    background: linear-gradient(160deg,#ffffff 0%,#f0ece3 100%);
    border-bottom: 1px solid #d4c8b0;
    padding: 40px 48px 32px; position: relative; overflow: hidden;
}
.hero-header::before {
    content: ''; position: absolute; top: -60px; right: -60px;
    width: 240px; height: 240px;
    background: radial-gradient(circle,rgba(200,169,110,0.15) 0%,transparent 70%);
    border-radius: 50%;
}
.hero-eyebrow {
    font-family: 'DM Sans', sans-serif; font-size: 11px; font-weight: 600;
    letter-spacing: 2.5px; text-transform: uppercase; color: #c8a96e; margin-bottom: 8px;
}
.hero-title {
    font-family: 'Playfair Display', serif !important; font-size: 38px !important;
    font-weight: 700 !important; color: #1a2f4a !important;
    letter-spacing: -0.5px; line-height: 1.2; margin: 0 0 8px !important;
}
.hero-sub {
    font-family: 'Source Serif 4', serif; font-size: 16px;
    color: #5a6b7a; font-weight: 300; margin: 0 0 20px !important;
}
.badges { display: flex; gap: 10px; flex-wrap: wrap; }
.badge {
    background: #1e3a5f; color: #e8d5a3;
    padding: 4px 14px; border-radius: 20px;
    font-size: 11px; font-weight: 600; letter-spacing: 1px; text-transform: uppercase;
}
.badge.gold { background: #c8a96e; color: #1a2f4a; }

.instr-card {
    background: #ffffff; border: 1px solid #e2ddd4; border-radius: 12px;
    padding: 20px 24px; margin: 0 0 18px;
    box-shadow: 0 2px 8px rgba(30,58,95,0.06);
}
.instr-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; }
.instr-item { font-family: 'DM Sans',sans-serif; font-size: 13.5px; color: #374151; line-height: 1.7; }
.instr-item strong { color: #1e3a5f; }

textarea, input[type=text] {
    font-family: 'Source Serif 4',serif !important; font-size: 15px !important;
    border: 1.5px solid #d4c8b0 !important; border-radius: 8px !important;
    background: #faf9f6 !important; color: #2d3748 !important;
    padding: 14px 16px !important; transition: border-color 0.2s !important;
}
textarea:focus, input[type=text]:focus {
    border-color: #1e3a5f !important;
    box-shadow: 0 0 0 3px rgba(30,58,95,0.08) !important; outline: none !important;
}

button.primary {
    background: linear-gradient(135deg,#1e3a5f 0%,#2d5a8f 100%) !important;
    color: white !important; font-family: 'DM Sans',sans-serif !important;
    font-weight: 600 !important; font-size: 15px !important;
    border: none !important; border-radius: 8px !important;
    padding: 14px 32px !important; cursor: pointer !important;
    transition: all 0.25s ease !important;
    box-shadow: 0 4px 14px rgba(30,58,95,0.28) !important;
}
button.primary:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 24px rgba(30,58,95,0.38) !important;
}

.tab-nav { border-bottom: 2px solid #e2ddd4 !important; }
.tab-nav button {
    font-family: 'DM Sans',sans-serif !important; font-weight: 500 !important;
    color: #5a6b7a !important; padding: 12px 22px !important;
}
.tab-nav button.selected {
    color: #1e3a5f !important; border-bottom: 3px solid #c8a96e !important;
    font-weight: 700 !important;
}

.prose-out {
    font-family: 'Source Serif 4',serif !important;
    font-size: 15px !important; line-height: 1.85 !important; color: #2d3748 !important;
}
.footer-bar {
    background: #1e3a5f; color: #94a3b8; text-align: center;
    padding: 18px 24px; font-family: 'DM Sans',sans-serif; font-size: 12px;
    letter-spacing: 0.5px; margin-top: 28px; border-radius: 0 0 12px 12px;
}
.footer-bar span { color: #c8a96e; font-weight: 600; }
"""

MARQUEE_HTML = """
<div class='marquee-wrapper'>
  <div class='marquee-content'>
    <span>📚 AI Literature Review Agent
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; Powered by LLaMA 3.1 8B
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; Research Intelligence Platform
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; Aigenthix
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; Academic AI Assistant
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; 6-Step Agent Pipeline
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; Groq Cloud</span>
    <span>📚 AI Literature Review Agent
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; Powered by LLaMA 3.1 8B
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; Research Intelligence Platform
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; Aigenthix
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; Academic AI Assistant
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; 6-Step Agent Pipeline
      &nbsp;<span class='mq-dot'>◆</span>&nbsp; Groq Cloud</span>
  </div>
</div>
"""

HERO_HTML = """
<div class='hero-header'>
  <p class='hero-eyebrow'>Aigenthix Research Intelligence</p>
  <h1 class='hero-title'>AI Literature Review Agent</h1>
  <p class='hero-sub'>A world-class research scientist &amp; academic reviewer powered by LLaMA 3.1 —
  analyzing, synthesizing, and reviewing research papers with scholarly precision.</p>
  <div class='badges'>
    <span class='badge'>LLaMA 3.1 8B</span>
    <span class='badge'>Groq Cloud</span>
    <span class='badge gold'>6-Step Agent Pipeline</span>
    <span class='badge'>PDF &amp; TXT Support</span>
    <span class='badge'>Comparative Analysis</span>
    <span class='badge'>Research Gaps Detection</span>
  </div>
</div>
"""

INSTRUCTIONS_HTML = """
<div class='instr-card'>
  <p style='font-family:DM Sans,sans-serif;font-size:11px;font-weight:600;letter-spacing:2px;text-transform:uppercase;color:#c8a96e;margin-bottom:10px;'>How to Use</p>
  <div class='instr-grid'>
    <div class='instr-item'>
      <strong>Step 1:</strong> Enter your research topic or problem statement below.<br>
      <strong>Step 2:</strong> Upload 1-10 research papers (PDF or TXT format).
    </div>
    <div class='instr-item'>
      <strong>Step 3:</strong> Or paste paper abstracts/full text in the expander.<br>
      <strong>Step 4:</strong> Click Generate and explore the 4 result tabs.
    </div>
  </div>
</div>
"""

FOOTER_HTML = """
<div class='footer-bar'>
  Built by <span>Aigenthix</span> &nbsp;|&nbsp;
  Powered by <span>LLaMA 3.1 8B (Groq)</span> &nbsp;|&nbsp;
  AI Literature Review Agent &nbsp;|&nbsp; © 2025 All Rights Reserved
</div>
"""

print('✅ UI assets ready')


## 🚀 Step 10: Launch Gradio Application


In [ ]:
with gr.Blocks(
    css=CUSTOM_CSS,
    title='AI Literature Review Agent | Aigenthix',
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.blue,
        secondary_hue=gr.themes.colors.amber,
        neutral_hue=gr.themes.colors.slate,
        font=[gr.themes.GoogleFont('DM Sans'), 'sans-serif'],
    )
) as app:

    # ── Marquee ──
    gr.HTML(MARQUEE_HTML)

    # ── Hero ──
    gr.HTML(HERO_HTML)

    with gr.Column():

        # Instructions
        gr.HTML(INSTRUCTIONS_HTML)

        # ── Inputs ──
        topic_input = gr.Textbox(
            label='Research Topic / Problem Statement',
            placeholder='e.g., Large Language Models for Medical Diagnosis and Clinical Decision Support...',
            lines=2, max_lines=5,
        )

        file_input = gr.File(
            label='Upload Research Papers (PDF or TXT)',
            file_count='multiple',
            file_types=['.pdf', '.txt', '.md'],
        )

        with gr.Accordion('📝 Paste Paper Text / Abstracts (Optional)', open=False):
            extra_text = gr.Textbox(
                label='Paper Content — separate multiple papers with ---',
                placeholder='Paste paper abstract or full text...\n---\nPaste another paper here...',
                lines=10, max_lines=30,
            )

        with gr.Row():
            submit_btn = gr.Button(
                '🔬 Generate Literature Review',
                variant='primary', scale=3
            )
            clear_btn = gr.ClearButton(value='↺ Clear All', scale=1)

        # ── Example Topics ──
        gr.Examples(
            examples=[
                ['Large Language Models for Medical Diagnosis and Clinical Decision Support'],
                ['Transformer Architectures in Computer Vision: A Comprehensive Survey'],
                ['Federated Learning for Privacy-Preserving Machine Learning Applications'],
                ['Reinforcement Learning from Human Feedback (RLHF) in Foundation Models'],
                ['Graph Neural Networks for Drug Discovery and Molecular Property Prediction'],
                ['Explainable AI in Manufacturing Quality Inspection and Defect Detection'],
            ],
            inputs=[topic_input],
            label='Quick Start Examples',
        )

    # ── Divider ──
    gr.HTML('<div style="margin:24px 0;"><hr style="border:none;border-top:2px solid #e2ddd4;"></div>')
    gr.HTML('<p style="font-family:DM Sans,sans-serif;font-size:11px;font-weight:600;letter-spacing:2px;text-transform:uppercase;color:#c8a96e;margin-bottom:12px;">Analysis Results</p>')

    # ── Output Tabs ──
    with gr.Tabs():

        with gr.TabItem('🧠 Summary & Synthesis'):
            gr.HTML('<p style="font-family:DM Sans,sans-serif;font-size:13px;color:#64748b;margin:8px 0 12px;">Topic analysis, cross-paper synthesis, research gaps, and paper rankings.</p>')
            summary_out = gr.Markdown(
                value='*Results will appear here after generating the review...*',
                elem_classes=['prose-out']
            )

        with gr.TabItem('📄 Paper-wise Analysis'):
            gr.HTML('<p style="font-family:DM Sans,sans-serif;font-size:13px;color:#64748b;margin:8px 0 12px;">Individual analysis for each uploaded paper: contributions, methodology, results, strengths, limitations, and relevance score.</p>')
            papers_out = gr.Markdown(
                value='*Individual paper analyses will appear here...*',
                elem_classes=['prose-out']
            )

        with gr.TabItem('✍️ Final Literature Review'):
            gr.HTML('<p style="font-family:DM Sans,sans-serif;font-size:13px;color:#64748b;margin:8px 0 12px;">Publication-ready academic literature review. Copy and paste directly into your research document.</p>')
            review_out = gr.Markdown(
                value='*The complete academic literature review will appear here...*',
                elem_classes=['prose-out']
            )

        with gr.TabItem('📊 Comparative Table'):
            gr.HTML('<p style="font-family:DM Sans,sans-serif;font-size:13px;color:#64748b;margin:8px 0 12px;">Side-by-side comparison of all analyzed papers with color-coded relevance scores.</p>')
            table_out = gr.HTML(
                value="<p style='color:#94a3b8;font-family:DM Sans,sans-serif;'>Comparative table will appear here after generation...</p>"
            )

    # ── Footer ──
    gr.HTML(FOOTER_HTML)

    # ── Events ──
    all_outputs = [summary_out, papers_out, review_out, table_out]

    submit_btn.click(
        fn=run_literature_review_agent,
        inputs=[topic_input, file_input, extra_text],
        outputs=all_outputs,
        show_progress=True,
    )
    clear_btn.add([topic_input, file_input, extra_text] + all_outputs)


print('🚀 Launching AI Literature Review Agent...')
app.launch(
    share=True,
    debug=False,
    show_error=True,
    inbrowser=False,
)
